# Etapa 2 — Preparação e Processamento de Dados

**Objetivo:** Demonstrar visualmente e numericamente o fluxo de transformação dos dados brutos até a matriz de entrada da MLP.

Este notebook espelha a lógica implementada em `src/data/loader.py`, `src/data/splitter.py` e `src/data/preprocessor.py`.

**Fluxo demonstrado:**
1. Carregamento bruto $\rightarrow$ Filtragem (S.PAULO + Roubo/Furto).
2. Consolidação de Boletins (várias linhas $\rightarrow$ 1 BO).
3. Divisão Estratificada (70% Treino, 15% Validação, 15% Teste).
4. Pré-processamento (Derivados $\rightarrow$ OneHot $\rightarrow$ Scaling).

## 1. Importações e Configurações

In [1]:
# Importa pandas para manipulação de DataFrames.
import pandas as pd
# Importa numpy para operações numéricas.
import numpy as np
# Importa a interface de gráficos do matplotlib.
import matplotlib.pyplot as plt
# Importa as funções de pipeline do projeto.
from src.data.loader import carregar_e_preparar
from src.data.splitter import preparar_split_2025
from src.data.preprocessor import preparar_features

# Configura o pandas para mostrar todas as colunas em tabelas largas.
pd.set_option('display.max_columns', None)
print('Importações concluídas.')

Importações concluídas.


## 2. Carregamento e Filtragem

In [2]:
# Carrega dados de 2025 aplicando filtros de Município (S.PAULO) e Rubrica (Roubo/Furto).
df_2025 = carregar_e_preparar(ano=2025)

# Exibe as primeiras linhas para conferir as colunas filtradas e a coluna 'target'.
print('Exemplo de dados filtrados (S.PAULO + Roubo/Furto):')
display(df_2025.head())

# Imprime a contagem final de registros após a filtragem.
print(f'Total de registros filtrados: {df_2025.shape[0]:,}')

Exemplo de dados filtrados (S.PAULO + Roubo/Furto):


,ID_DELEGACIA,NOME_DEPARTAMENTO,NOME_SECCIONAL,NOME_DELEGACIA,NOME_MUNICIPIO,ANO_BO,NUM_BO,VERSAO,NOME_DEPARTAMENTO_CIRC,NOME_SECCIONAL_CIRC,NOME_DELEGACIA_CIRC,NOME_MUNICIPIO_CIRC,DATA_OCORRENCIA_BO,HORA_OCORRENCIA,DESCRICAO_APRESENTACAO,DATAHORA_REGISTRO_BO,DATA_COMUNICACAO_BO,DATAHORA_IMPRESSAO_BO,DESCR_PERIODO,AUTORIA_BO,FLAG_INTOLERANCIA,TIPO_INTOLERANCIA,FLAG_FLAGRANTE,FLAG_STATUS,DESC_LEI,FLAG_ATO_INFRACIONAL,RUBRICA,DESCR_CONDUTA,DESDOBRAMENTO,CIRCUNSTANCIA,DESCR_TIPOLOCAL,DESCR_SUBTIPOLOCAL,CIDADE,BAIRRO,CEP,LOGRADOURO_VERSAO,LOGRADOURO,NUMERO_LOGRADOURO,LATITUDE,LONGITUDE,CONT_OBJETO,DESCR_MODO_OBJETO,DESCR_TIPO_OBJETO,DESCR_SUBTIPO_OBJETO,DESCR_UNIDADE,QUANTIDADE_OBJETO,MARCA_OBJETO,FLAG_BLOQUEIO,FLAG_DESBLOQUEIO,MES_REGISTRO_BO,ANO_REGISTRO_BO,CMD,BTL,CIA,CD_IBGE,target
0,900020,DIPOL - DEPTO DE INTELIGENCIA,DELEGACIA ELETRONICA,DELEGACIA ELETRONICA,S.PAULO,2025,AA1691,1,DECAP,DEL.SEC.1º CENTRO,01º D.P. SE,S.PAULO,2024-12-31,18:30:00,Pela Parte Interessada,2025-01-01,2025-01-01,2025-01-01,NaN,D,N,NaN,N,CONSUMADO,Código Penal,N,Roubo (art. 157),Outros,NaN,NaN,Via Pública,Via Pública,S.PAULO,LIBERDADE,1514001.0,1,RUA GLICÉRIO,671.0,-23.558166,-46.630172,1008550122,Subtraído,Telecomunicação,Telefone Celular,NaN,1,Xiaomi,S,N,1,2025,CPA/M-1,7ºBPM/M,3ªCIA_4ªCIA,3550308,1
1,900021,DIPOL - DEPTO DE INTELIGENCIA,DELEGACIA ELETRONICA,DELEGACIA ELETRONICA 1,S.PAULO,2025,AA2064,1,DECAP,DEL.SEC.1º CENTRO,01º D.P. SE,S.PAULO,2024-12-30,15:00:00,Pela Parte Interessada,2025-01-01,2025-01-01,2025-01-01,NaN,D,N,NaN,N,CONSUMADO,Código Penal,N,Furto (art. 155),Outros,NaN,NaN,Via Pública,Via Pública,S.PAULO,SE,1021200.0,1,RUA VINTE E CINCO DE MARCO,1081.0,-23.540202,-46.632154,1008550328,Subtraído,Telecomunicação,Telefone Celular,NaN,1,Apple,S,N,1,2025,CPA/M-1,7ºBPM/M,3ªCIA_4ªCIA,3550308,0
2,900020,DIPOL - DEPTO DE INTELIGENCIA,DELEGACIA ELETRONICA,DELEGACIA ELETRONICA,S.PAULO,2025,AA2458,1,DECAP,DEL.SEC.1º CENTRO,01º D.P. SE,S.PAULO,2025-01-01,04:30:00,Pela Parte Interessada,2025-01-01,2025-01-01,2025-01-01,NaN,D,N,NaN,N,CONSUMADO,Código Penal,N,Roubo (art. 157),Outros,NaN,NaN,Via Pública,Via Pública,S.PAULO,BELA VISTA,1314000.0,1,RUA SANTO ANTÔNIO,54.0,-23.549928,-46.640538,1008550572,Subtraído,Telecomunicação,Telefone Celular,NaN,1,Motorola,S,N,1,2025,CPA/M-1,7ºBPM/M,3ªCIA_4ªCIA,3550308,1
3,900020,DIPOL - DEPTO DE INTELIGENCIA,DELEGACIA ELETRONICA,DELEGACIA ELETRONICA,S.PAULO,2025,AA2520,1,DECAP,DEL.SEC.1º CENTRO,01º D.P. SE,S.PAULO,2025-01-01,NaN,Pela Parte Interessada,2025-01-01,2025-01-01,2025-01-01,De madrugada,D,N,NaN,N,CONSUMADO,Código Penal,N,Furto (art. 155),Outros,NaN,NaN,Via Pública,Via Pública,S.PAULO,LIBERDADE,1507020.0,1,RUA JUNQUEIRA FREIRE,231.0,-23.559358,-46.628774,1008550622,Subtraído,Telecomunicação,Telefone Celular,NaN,1,Samsung,S,N,1,2025,CPA/M-1,7ºBPM/M,3ªCIA_4ªCIA,3550308,0
4,900020,DIPOL - DEPTO DE INTELIGENCIA,DELEGACIA ELETRONICA,DELEGACIA ELETRONICA,S.PAULO,2025,AA2537,1,DECAP,DEL.SEC.1º CENTRO,01º D.P. SE,S.PAULO,2024-12-29,NaN,Pela Parte Interessada,2025-01-01,2025-01-01,2025-01-01,A noite,D,N,NaN,N,CONSUMADO,Código Penal,N,Furto (art. 155),Outros,NaN,NaN,Terminal/Estação,Metroviário e Ferroviário Metropolitano,S.PAULO,LIBERDADE,13104098.0,1,RUA DA GLÓRIA,758.0,-23.559892,-46.633391,1008550640,Subtraído,Telecomunicação,Telefone Celular,NaN,1,Apple,S,N,1,2025,CPA/M-1,7ºBPM/M,3ªCIA_4ªCIA,3550308,0


Total de registros filtrados: 183,798


## 3. Consolidação de Boletins

Um mesmo número de boletim (`NUM_BO`) pode aparecer em várias linhas se houver múltiplos celulares subtraídos. Consolidados em uma única linha por BO.

In [3]:
# Importa a função de consolidação do splitter.
from src.data.splitter import consolidar_boletins

# Consolida os dados: se houver um roubo no BO, o BO inteiro é marcado como roubo (target=1).
df_consolidado = consolidar_boletins(df_2025)

# Compara o tamanho antes e depois da consolidação.
print(f'Linhas antes da consolidação: {df_2025.shape[0]:,}')
print(f'Linhas após a consolidação (por NUM_BO): {df_consolidado.shape[0]:,}')
print(f'Redução de {100 * (1 - df_consolidado.shape[0]/df_2025.shape[0]):.1f}%')

# Exibe as primeiras linhas do DataFrame consolidado.
display(df_consolidado.head())

Linhas antes da consolidação: 183,798
Linhas após a consolidação (por NUM_BO): 161,614
Redução de 12.1%


,NUM_BO,target,DESCR_TIPOLOCAL,DESCR_SUBTIPOLOCAL,BAIRRO,DESCR_PERIODO,HORA_OCORRENCIA,DATA_OCORRENCIA_BO
0,AA0024,1,Via Pública,Via Pública,RIO PEQUENO,missing,21:15:00,2024-12-31
1,AA0058,0,Estacionamento/Garagem,Garagem ou abrigo de residência,VILA MATILDE,missing,22:25:00,2024-12-31
2,AA0073,0,Via Pública,Via Pública,CENTRO,missing,15:51:00,2024-12-31
3,AA0079,1,Via Pública,Via Pública,JARDIM ITAPURA,missing,00:45:00,2024-12-31
4,AA0085,0,Via Pública,Via Pública,CERQUEIRA CÉSAR,A noite,missing,2024-12-31


## 4. Divisão Estratificada (70/15/15)

In [4]:
# Divide os dados consolidados em treino, validação e teste (2025).
train, val, test_2025 = preparar_split_2025(df_2025)

# Imprime o tamanho de cada conjunto.
print(f'Treino: {train.shape[0]:,}')
print(f'Validação: {val.shape[0]:,}')
print(f'Teste (2025): {test_2025.shape[0]:,}')

# Verifica se a proporção do target foi mantida (estratificação).
print('\nDistribuição do Target (Média de Roubo):')
print(f'Treino: {train["target"].mean():.4f}')
print(f'Validação: {val["target"].mean():.4f}')
print(f'Teste: {test_2025["target"].mean():.4f}')

Treino: 113,129
Validação: 24,242
Teste (2025): 24,243

Distribuição do Target (Média de Roubo):
Treino: 0.3693
Validação: 0.3693
Teste: 0.3693


## 5. Pré-processamento e Transformação

Demonstra a transformação de dados categóricos (Bairro, Local) em vetores numéricos (OneHot) e a escala das numéricas.

In [5]:
# Fit: ajusta o preprocessor apenas nos dados de treino.
X_train, y_train, preprocessor = preparar_features(train, fit=True)

# Transform: aplica as mesmas regras no conjunto de validação.
X_val, y_val, _ = preparar_features(val, preprocessor=preprocessor, fit=False)

# Exibe a dimensão final da matriz de entrada da MLP.
print(f'Dimensão de X_train: {X_train.shape}')
print(f'Número de features geradas: {X_train.shape[1]}')

# Visualiza a matriz resultante (primeiras 5 linhas e 10 colunas).
print('\nPreview da matriz X_train (amostra):')
display(X_train[:5, :10])

Dimensão de X_train: (113129, 1914)
Número de features geradas: 1914

Preview da matriz X_train (amostra):


array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

## 6. Conclusões

1. **Filtragem**: O recorte para São Paulo e rubricas de roubo/furto é aplicado corretamente via `loader.py`.
2. **Consolidação**: A agregação por `NUM_BO` remove a duplicidade de itens, evitando que o modelo aprenda a frequência de itens por BO em vez da natureza do crime.
3. **Estratificação**: A proporção de roubos é consistente entre treino, validação e teste.
4. **Dimensionalidade**: O OneHotEncoding de `BAIRRO` é a principal fonte de expansão de colunas, resultando em ~1.9k features.